# Prism Winner Validation — Longer Training
**Runtime → A100**

Runs the sweep winner (rand_07 config) for 2000 steps to see if the 2.74x advantage holds.
Also runs ortho baseline at 2000 steps for comparison.
Each saves to disk immediately.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/prism'):
    subprocess.run(['rm', '-rf', '/content/prism'])
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism
!pip install -q transformers datasets scipy
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Ready.')

In [ ]:
# Cell 2: Run PRISM with winning config (2000 steps)
import sys, os, json, gc, time, warnings
import torch
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn

install_signal_handlers()
device = 'cuda'

with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)
print('Extracting pretrained directions...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')
init_fn = make_hybrid_init_fn(
    extracted['spectra_coeffs'], dirs,
    lam=1.0, align_mode='UV', align_strength=0.75  # winner: stronger alignment
)
gc.collect()

config = TrainConfig(
    max_steps=2000,
    eval_steps=[100, 250, 500, 750, 1000, 1500, 2000],
    warmup_steps=300,
    log_every=50,
    lr=9.38e-5,            # 1.5x base LR
    spike_skip_mult=50.0,  # the stabilizer
    grad_clip=1.0,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

print('='*60)
print('  PRISM WINNER: align=0.75, LR=1.5x, spike_skip=50')
print('='*60)
t0 = time.time()
result = train(config, init_fn=init_fn, init_name='prism_winner', verbose=True)
elapsed = time.time() - t0
print(f'\nDone in {elapsed:.0f}s — Final PPL: {result["final_ppl"]:.1f}')

data = {
    'name': 'prism_winner_2k',
    'config': 'align=0.75, lr=1.5x, spike_skip=50, warmup=300',
    'final_ppl': result['final_ppl'],
    'elapsed': elapsed,
    'checkpoints': {str(k): v for k, v in result['checkpoints'].items()},
    'gpu': torch.cuda.get_device_name(0),
    'steps': 2000,
}
with open('/content/prism_winner_2k.json', 'w') as f:
    json.dump(data, f, indent=2)
print('SAVED: /content/prism_winner_2k.json')
_clear_memory(device); gc.collect()

In [ ]:
# Cell 3: Run ORTHO baseline (2000 steps)
import sys, os, json, gc, time, warnings
import torch
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn

install_signal_handlers()
device = 'cuda'

config = TrainConfig(
    max_steps=2000,
    eval_steps=[100, 250, 500, 750, 1000, 1500, 2000],
    warmup_steps=300,
    log_every=50,
    lr=6.25e-5,
    grad_clip=1.0,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

print('='*60)
print('  ORTHO BASELINE: lr=1x')
print('='*60)
t0 = time.time()
result = train(config, init_fn=make_init_fn('orthogonal'),
               init_name='ortho_baseline', verbose=True)
elapsed = time.time() - t0
print(f'\nDone in {elapsed:.0f}s — Final PPL: {result["final_ppl"]:.1f}')

data = {
    'name': 'ortho_baseline_2k',
    'final_ppl': result['final_ppl'],
    'elapsed': elapsed,
    'checkpoints': {str(k): v for k, v in result['checkpoints'].items()},
    'gpu': torch.cuda.get_device_name(0),
    'steps': 2000,
}
with open('/content/ortho_baseline_2k.json', 'w') as f:
    json.dump(data, f, indent=2)
print('SAVED: /content/ortho_baseline_2k.json')
_clear_memory(device); gc.collect()

In [ ]:
# Cell 4: Compare
import json
p = json.load(open('/content/prism_winner_2k.json'))
o = json.load(open('/content/ortho_baseline_2k.json'))
print('='*50)
print(f'GPU: {p["gpu"]}')
print(f'\n{"Step":>6s}  {"Ortho":>8s}  {"Prism":>8s}  {"Ratio":>7s}')
print('-' * 35)
for s in ['100','250','500','750','1000','1500','2000']:
    ov = o['checkpoints'].get(s)
    pv = p['checkpoints'].get(s)
    if ov and pv:
        print(f'{s:>6s}  {ov:>8.1f}  {pv:>8.1f}  {ov/pv:>6.2f}x')
r = o['final_ppl'] / p['final_ppl']
print(f'\nFinal: ortho={o["final_ppl"]:.1f}  prism={p["final_ppl"]:.1f}  ratio={r:.2f}x')
print(f'\nKey question: does the advantage hold, shrink, or collapse past 750 steps?')

In [ ]:
# Cell 5: Download
from google.colab import files
for f in ['/content/prism_winner_2k.json', '/content/ortho_baseline_2k.json']:
    try: files.download(f)
    except: print(f'{f} not found')